In [3]:
import numpy as np
import pdfplumber
import pandas as pd
import re

In [ ]:
def cargar_pdf(ruta_pdf: str):
    f = open(ruta_pdf, "rb")
    pdf = pdfplumber.open(f)
    return pdf, f

def extraer_texto_paginas(pdf) -> str:
    texto = ""
    for p in pdf.pages:
        contenido = p.extract_text()
        if contenido:
            texto += contenido + "\n"
    return texto

def limpiar_lineas(texto: str) -> list:
    lineas = [line.strip() for line in texto.split("\n") if line.strip()]
    return lineas

def extraer_transacciones(lineas: list) -> pd.DataFrame:
    patron = re.compile(
        r"(\d{2}/\d{2})\s+(.*?)\s+(\d{1,3}(?:\.\d{3})*,\d{2})"
    )

    registros = []
    for linea in lineas:
        match = patron.search(linea)
        if match:
            fecha, establecimiento, valor = match.groups()
            registros.append({
                "fecha": fecha,
                "establecimiento": establecimiento,
                "valor": float(valor.replace(".", "").replace(",", "."))
            })

    return pd.DataFrame(registros)

def cargar_estado_cuenta(ruta_pdf: str) -> pd.DataFrame:
    pdf, f = cargar_pdf(ruta_pdf)
    texto = extraer_texto_paginas(pdf)
    lineas = limpiar_lineas(texto)
    df = extraer_transacciones(lineas)
    pdf.close()
    f.close()
    return df

ruta = "D:/DRVACAE/python_projects/presupuesto/data/diners.pdf"

df = cargar_estado_cuenta(ruta)
print(df)
df.to_excel("output/estado_cuenta.xlsx", index=False)

    fecha                              establecimiento    valor
0   30/05  8119543 DIFERIDO INTERNACIONAL ONLINE (3/6)   671.70
1   30/05  8119546 DIFERIDO INTERNACIONAL ONLINE (3/6)   327.12
2   17/07  8169587 DIFERIDO INTERNACIONAL ONLINE (2/6)   122.28
3   28/11   994746 PTP - UNIVERSIDAD INTERNACIO (9/24)   110.80
4   20/07                    9393961 EL CORTE INGLES M   101.93
5   20/07         9393961 CARGO CONSUMO EN EL EXTERIOR     1.70
6   20/07                    9393962 GRAN VIA MADRID M   535.14
7   20/07         9393962 CARGO CONSUMO EN EL EXTERIOR     1.70
8   21/07            9561089 FARMACIA BERNABE DEL AMOM    16.40
9   16/07                         9024935 UBER RIDES S     2.33
10  16/07            9024935 RET IVA SERV DIGITAL 100%     0.35
11  16/07                         9024936 UBER RIDES S     3.03
12  16/07            9024936 RET IVA SERV DIGITAL 100%     0.45
13  17/07                         9024937 UBER RIDES S     4.07
14  17/07            9024937 RET IVA SER